In [2]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for saving files
import os
import sys

project_path = r'C:\Users\VISHNU\Downloads\nifty100_project'
sys.path.append(project_path)
os.chdir(project_path)

# Create radar charts folder
os.makedirs('reports/radar_charts', exist_ok=True)

from src.etl.loader import load_all_data

data        = load_all_data()
peer_groups = data['peer_groups']
sectors     = data['sectors']

# Load peer percentiles from database
conn = sqlite3.connect('data/nifty100.db')
peer_percentiles = pd.read_sql_query(
    "SELECT * FROM peer_percentiles", conn
)
ratios = pd.read_sql_query(
    "SELECT * FROM financial_ratios_computed", conn
)
conn.close()

print(f"Peer percentiles: {peer_percentiles.shape}")
print(f"Ratios: {ratios.shape}")
print("Setup complete!")

Loading all datasets...

Dataset Summary:
  Dataset                Rows   Cols
  -----------------------------------
  profitandloss          1164     15
  balancesheet           1165     13
  cashflow               1152      7
  companies                92     12
  analysis                 20      6
  documents              1585      4
  prosandcons              16      4
  sectors                  92      6
  market_cap              552      9
  financial_ratios       1184     16
  peer_groups              56      4

All datasets loaded and cleaned successfully!
Peer percentiles: (560, 6)
Ratios: (1159, 43)
Setup complete!


In [3]:
def plot_radar_chart(company_id, group_name, company_values,
                     group_avg_values, metrics, output_dir):
    """
    Generates a radar/spider chart for a company vs peer group average.

    Args:
        company_id:       NSE ticker
        group_name:       Peer group name
        company_values:   List of percentile ranks for the company
        group_avg_values: List of average percentile ranks for the group
        metrics:          List of metric names for each axis
        output_dir:       Directory to save PNG files
    """
    N = len(metrics)
    if N == 0:
        return

    # Compute angle for each axis
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]  # Close the polygon

    # Complete the loops
    company_values  = list(company_values) + [company_values[0]]
    group_avg_values = list(group_avg_values) + [group_avg_values[0]]

    # Short metric labels for readability
    labels = [
        m.replace('return_on_equity_pct',    'ROE')
         .replace('return_on_capital_pct',   'ROCE')
         .replace('net_profit_margin_pct',   'NPM')
         .replace('debt_to_equity',          'D/E')
         .replace('free_cash_flow_cr',       'FCF')
         .replace('net_profit_cagr_5yr',     'PAT\nCAGR')
         .replace('sales_cagr_5yr',          'Rev\nCAGR')
         .replace('eps_cagr_5yr',            'EPS\nCAGR')
         .replace('interest_coverage',       'ICR')
         .replace('asset_turnover',          'Asset\nTO')
        for m in metrics
    ]

    # Plot
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

    # Company polygon
    ax.plot(angles, company_values, 'o-', linewidth=2,
            color='#1F4E79', label=company_id)
    ax.fill(angles, company_values, alpha=0.25, color='#1F4E79')

    # Peer group average (dashed)
    ax.plot(angles, group_avg_values, 'o--', linewidth=1.5,
            color='#ED7D31', label='Peer Avg')
    ax.fill(angles, group_avg_values, alpha=0.10, color='#ED7D31')

    # Axis labels
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, size=10)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.50, 0.75, 1.0])
    ax.set_yticklabels(['25th', '50th', '75th', '100th'], size=8)
    ax.grid(color='grey', linestyle='--', linewidth=0.5, alpha=0.7)

    # Title and legend
    ax.set_title(
        f"{company_id} — {group_name}\nPeer Percentile Ranks",
        size=13, fontweight='bold', pad=20
    )
    ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

    # Save
    filename = f"{company_id}_radar.png"
    filepath = os.path.join(output_dir, filename)
    plt.tight_layout()
    plt.savefig(filepath, dpi=100, bbox_inches='tight')
    plt.close()

    return filepath

print("Radar chart function ready!")

Radar chart function ready!


In [4]:
RADAR_METRICS = [
    'return_on_equity_pct',
    'return_on_capital_pct',
    'net_profit_margin_pct',
    'debt_to_equity',
    'free_cash_flow_cr',
    'net_profit_cagr_5yr',
    'sales_cagr_5yr',
    'interest_coverage',
]

generated = []
failed    = []

for group_name in peer_percentiles['peer_group_name'].unique():
    group_data = peer_percentiles[
        peer_percentiles['peer_group_name'] == group_name
    ]

    # Get metrics available for this group
    available_metrics = [
        m for m in RADAR_METRICS
        if m in group_data['metric'].values
    ]

    if len(available_metrics) < 3:
        continue

    # Compute group average percentile per metric
    group_avg = group_data.groupby('metric')['percentile_rank'].mean()

    # Generate chart for each company in the group
    companies = group_data['company_id'].unique()

    for company_id in companies:
        company_data = group_data[
            group_data['company_id'] == company_id
        ].set_index('metric')

        # Get values for available metrics
        company_vals = []
        avg_vals     = []

        for metric in available_metrics:
            c_val = company_data.loc[metric, 'percentile_rank'] \
                    if metric in company_data.index else 0.5
            a_val = group_avg.get(metric, 0.5)
            company_vals.append(float(c_val) if pd.notna(c_val) else 0.5)
            avg_vals.append(float(a_val) if pd.notna(a_val) else 0.5)

        try:
            filepath = plot_radar_chart(
                company_id, group_name,
                company_vals, avg_vals,
                available_metrics,
                'reports/radar_charts'
            )
            generated.append(company_id)
        except Exception as e:
            failed.append((company_id, str(e)))

print(f"Radar charts generated: {len(generated)}")
print(f"Failed: {len(failed)}")
if failed:
    print("Failed companies:", failed)
print(f"\nCharts saved to: reports/radar_charts/")

Radar charts generated: 56
Failed: 0

Charts saved to: reports/radar_charts/


In [5]:
ANALYSIS_YEAR = '2024-03'

# Get latest ratios with sector info
latest = ratios[ratios['year'] == ANALYSIS_YEAR].copy()

numeric_cols = [
    'return_on_equity_pct', 'return_on_capital_pct',
    'net_profit_margin_pct', 'debt_to_equity',
    'free_cash_flow_cr', 'sales_cagr_5yr',
    'net_profit_cagr_5yr', 'interest_coverage', 'asset_turnover'
]
for col in numeric_cols:
    if col in latest.columns:
        latest[col] = pd.to_numeric(latest[col], errors='coerce')

# Merge with sectors
sector_df = pd.merge(
    latest,
    sectors[['company_id', 'broad_sector', 'sub_sector']],
    on='company_id', how='left'
)

# Compute sector aggregates
sector_agg = sector_df.groupby('broad_sector').agg(
    company_count        = ('company_id', 'count'),
    median_roe           = ('return_on_equity_pct', 'median'),
    median_roce          = ('return_on_capital_pct', 'median'),
    median_npm           = ('net_profit_margin_pct', 'median'),
    median_de            = ('debt_to_equity', 'median'),
    median_fcf           = ('free_cash_flow_cr', 'median'),
    median_rev_cagr_5yr  = ('sales_cagr_5yr', 'median'),
    median_pat_cagr_5yr  = ('net_profit_cagr_5yr', 'median'),
).round(2).reset_index()

print("Sector Analytics — Median KPIs:")
print(sector_agg.to_string(index=False))

Sector Analytics — Median KPIs:
          broad_sector  company_count  median_roe  median_roce  median_npm  median_de  median_fcf  median_rev_cagr_5yr  median_pat_cagr_5yr
Communication Services              2        6.16         7.50       14.58       1.31     13829.5                15.14                19.23
Consumer Discretionary             14       24.20        18.20        9.70       0.05      1215.5                10.77                15.32
      Consumer Staples              7       20.07        32.64       14.60       0.14      2938.0                 8.69                12.10
                Energy             14       13.25         9.84       10.80       0.82      8644.0                10.72                12.82
            Financials             23       15.74         4.17       19.94       4.39     -2099.0                17.37                23.36
            Healthcare              6       15.32        18.50       17.97       0.06      1072.0                10.22          

In [6]:
conn = sqlite3.connect('data/nifty100.db')
sector_agg.to_sql('sector_analytics', conn,
                  if_exists='replace', index=False)

cursor = conn.cursor()
cursor.execute("SELECT COUNT(*) FROM sector_analytics")
count = cursor.fetchone()[0]
conn.close()

print(f"sector_analytics table saved — {count} sectors")

sector_analytics table saved — 10 sectors


In [7]:
print("Sprint 3 Day 4 — Summary:")
print(f"  Radar charts generated: {len(generated)}")
print(f"  Radar charts location:  reports/radar_charts/")
print()
print(f"  Sectors analysed:       {len(sector_agg)}")
print()
print("Top 3 sectors by median ROE:")
print(sector_agg.nlargest(3, 'median_roe')[
    ['broad_sector', 'median_roe', 'company_count']
].to_string(index=False))
print()
print("Top 3 sectors by median Revenue CAGR 5yr:")
print(sector_agg.nlargest(3, 'median_rev_cagr_5yr')[
    ['broad_sector', 'median_rev_cagr_5yr', 'company_count']
].to_string(index=False))

Sprint 3 Day 4 — Summary:
  Radar charts generated: 56
  Radar charts location:  reports/radar_charts/

  Sectors analysed:       10

Top 3 sectors by median ROE:
          broad_sector  median_roe  company_count
Consumer Discretionary       24.20             14
Information Technology       23.01              5
      Consumer Staples       20.07              7

Top 3 sectors by median Revenue CAGR 5yr:
          broad_sector  median_rev_cagr_5yr  company_count
            Financials                17.37             23
Communication Services                15.14              2
Information Technology                12.71              5
